# LLM Fine-tuning Notebook

LLM Fine-tuning Notebook for Llama-3.2-3B-Instruct with Unsloth

This notebook guides through the process of fine-tuning the Llama-3.2-3B-Instruct model
using the Unsloth library on Google Colab (T4 GPU).


# 1. Setup
Install necessary libraries and authenticate with Hugging Face.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import getpass
import os

repo_name = "llm-model-generation"
github_repo_url = "https://github.com/TsutsujiStudyTeam/llm-model-generation.git"

if not os.path.exists(repo_name):
    print(f"Cloning {repo_name} from GitHub...")
    github_token = getpass.getpass("Enter your GitHub Personal Access Token: ")
    
    # git config を使ってPATを設定
    !git config --global url."https://{github_token}@github.com/".insteadOf "https://github.com/"
    
    # 通常のgit cloneコマンドを実行
    !git clone {github_repo_url}
    
    %cd {repo_name}
else:
    print(f"{repo_name} already exists. Changing directory to {repo_name}...")
    %cd {repo_name}


In [ ]:
# Install Unsloth and other dependencies
!pip install "unsloth[colab-new]" peft accelerate bitsandbytes transformers trl huggingface_hub pyyaml


In [ ]:
# Authenticate with Hugging Face
# You will be prompted to enter your Hugging Face token.
notebook_login()


# 2. Data Preparation
Load and preprocess the training dataset.


# 3. Model Loading
Load the base language model and prepare it for LoRA fine-tuning.


# 4. Tokenizer (Already handled in Model Loading, but shown as a separate step for clarity)
The tokenizer is loaded along with the model.


# 5. Training
Fine-tune the model using the prepared dataset.


# 6. Save LoRA adapter
Save the trained LoRA adapter locally.


# 7. Upload to Hugging Face Hub
Upload the trained LoRA adapter to Hugging Face Hub.


# 8. Inference (Example - for testing in Colab)
Quick test of the fine-tuned model.


# 2. Run Fine-tuning ScriptExecute the main fine-tuning logic from `finetune_script.py`.

In [ ]:
!python training/finetune_script.py

# 3. Inference (Example - for testing in Colab)Quick test of the fine-tuned model by reloading the saved adapter.

In [ ]:
import torchfrom unsloth import FastLanguageModelfrom transformers import AutoTokenizer# Load parameters for model detailsimport yamlwith open("../params.yaml", "r") as f:    params = yaml.safe_load(f)hf_model_repo = params["hf_model_repo"]max_seq_length = params["max_seq_length"]# Reload the merged model for inference# Note: The script saves to 'lora_adapter_merged', so we load from there.model, tokenizer = FastLanguageModel.from_pretrained(    model_name = "lora_adapter_merged", # Load from local path    max_seq_length = max_seq_length,    dtype = None, # None for auto detection.    load_in_4bit = True, # Use 4bit quantization)FastLanguageModel.for_inference(model)messages = [    {"role": "user", "content": "私は犬が好きです。これを否定文に変換してください。"},]inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")outputs = model.generate(inputs, max_new_tokens=64, use_cache=True)print(tokenizer.decode(outputs[0], skip_special_tokens=True))